In [78]:
import flwr as fl
import torch
import os

# wrong

In [79]:
# BATCH_SIZE=32
# LEARNING_RATE = 0.01
# NUM_EPOCHS = 200
INPUT_SIZE = 4
OUTPUT_SIZE = 2
# HIDDEN_SIZE = 256
# NUM_LAYERS = 3

# target_feature_names = ['throughput', 'delay']
# RESULTS_DIR = "../results/"

# FUTURE_WIN_LEN = 5
# WINDOW_LEN = 15
# STRIDE = FUTURE_WIN_LEN
# FILTER_LEN = 5 

NUM_CLIENTS=7
NUM_ROUNDS=200 #100-200

In [80]:
client_resources = {"num_cpus":1,"num_gpus":1}

In [81]:
import math
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.metrics import r2_score


import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
from torch.optim import lr_scheduler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cpu


# hyp to tune

BATCH_SIZE,LEARNING_RATE,NUM_EPOCHS, HIDDEN_SIZE, NUM_LAYERS, FUTURE_WIN_LEN, WINDOW_LEN, FILTER_LEN

In [82]:
import pandas as pd

df = pd.read_csv('../ML-Aided-NR-V2X-QoS-Predictions/dataset.csv')
df.drop(columns='Unnamed: 0', inplace=True)
df.drop_duplicates(inplace = True)
# df.head()

In [83]:
cols = ['throughput', 'sinr', 'dlpathloss', 'delay']

# print(cols)
# print(len(cols))
delay_index = cols.index('delay')
tp_index = cols.index('throughput')
# print(delay_index, tp_index)

In [84]:
from sklearn.linear_model import LinearRegression
clean = df[['distance', 'pdr']].dropna()

model = LinearRegression()
model.fit(clean[['pdr']], clean[['distance']])
missing_dist = df['distance'].isnull()
predicted = model.predict(df.loc[missing_dist, ['pdr']])
df.loc[missing_dist, 'distance'] = predicted

In [85]:
clean = df[['jitter', 'delay']].dropna()
model = LinearRegression()
model.fit(clean[['delay']], clean[['jitter']])
missing_jitter = df['jitter'].isnull()
predicted = model.predict(df.loc[missing_jitter, ['delay']])
df.loc[missing_jitter, 'jitter'] = predicted

In [86]:
cols_to_smoothen = ['sinr', 'dlpathloss']
for col in cols_to_smoothen:
    df[col] = df[col].rolling(window=FILTER_LEN).mean()

df.bfill(inplace=True)
df.ffill(inplace=True)

# df.head(15)

In [87]:
def sliding_window_augmentation(df, cols, indices, future_window_len=1, L=100, s=10, add_noise=True, noise_level=0.02):
    X = df[cols].values
    T, D = X.shape

    n_windows = max(0, (T - L - future_window_len) // s + 1)

    if n_windows <= 0:
        return np.array([]), np.array([])

    num_targets = len(indices)
    windows = np.zeros((n_windows, L, D))
    targets = np.zeros((n_windows, future_window_len, num_targets))

    col_stds = df[cols].std().values

    for i in range(n_windows):
        start = i * s
        end_input = start + L
        end_target = end_input + future_window_len

        w = X[start:end_input].copy()
        
        target = X[end_input:end_target, indices].copy()

        if add_noise:
            input_noise = np.random.randn(*w.shape) * col_stds * noise_level 
            w += input_noise

            target_noise = np.random.randn(future_window_len, num_targets) * col_stds[indices] * noise_level
            target += target_noise

        windows[i] = w
        targets[i] = target

    return windows, targets

In [88]:
from torch.utils.data import Dataset, DataLoader

class QosDataset(Dataset):
    def __init__(self, windowed_data, target):
        self.X = torch.tensor(windowed_data, dtype=torch.float32)
        self.y = torch.tensor(target, dtype=torch.float32)

    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [89]:
def create_autoregressive_dataset(data, cols, indices, future_window_len=1, window_len = 100, stride = 10):
    windows, targets = sliding_window_augmentation(
        data, cols, indices, future_window_len=future_window_len, L = window_len, s = stride, add_noise=False
    )

    window_noisy, target_noisy = sliding_window_augmentation(
        data, cols, indices, future_window_len=future_window_len, L = window_len, s = stride, add_noise=True
    )

    if len(windows) > 0 and len(window_noisy) > 0:
        all_windows = np.concatenate([windows, window_noisy], axis=0)
        all_targets = np.concatenate([targets, target_noisy], axis=0)

    elif len(windows) > 0:
        all_windows = windows
        all_targets = targets
    else:
        raise ValueError("No valid windows created")

    # print("number of windows ", len(window_noisy))

    dataset = QosDataset(all_windows, all_targets)

    return dataset, all_windows, all_targets

In [90]:
indices = []
indices.append(cols.index('throughput'))
indices.append(cols.index('delay'))

# print(indices)

In [91]:
dataset, all_windows, all_targets = create_autoregressive_dataset(
    df, cols, indices, future_window_len=FUTURE_WIN_LEN, window_len=WINDOW_LEN, stride=STRIDE
)


In [92]:
import numpy as np
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

def create_federated_data(all_windows, all_targets, num_clients, batch_size, train_test_split_ratio = 0.8):
    client_loaders = []
    num_samples = len(all_windows)
    samples_per_client = num_samples // num_clients
        
    for i in range(num_clients):
        start_idx = i * samples_per_client
        end_idx = (i + 1) * samples_per_client if i < num_clients - 1 else num_samples
        
        client_windows_partition = all_windows[start_idx:end_idx]
        client_target_partition = all_targets[start_idx:end_idx]

        if len(client_windows_partition) < 2: 
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            client_windows_partition, client_target_partition, train_size=train_test_split_ratio, shuffle=False
        )

        scaler = MinMaxScaler()
        n_samples_train, n_timesteps, n_features = X_train.shape
        X_train_reshaped = X_train.reshape(-1, n_features)
        X_train_scaled_reshaped = scaler.fit_transform(X_train_reshaped)
        X_train_scaled = X_train_scaled_reshaped.reshape(n_samples_train, n_timesteps, n_features)

        n_samples_test = X_test.shape[0]
        X_test_reshaped = X_test.reshape(-1, n_features)
        X_test_scaled_reshaped = scaler.transform(X_test_reshaped)
        X_test_scaled = X_test_scaled_reshaped.reshape(n_samples_test, n_timesteps, n_features)

        target_scaler = MinMaxScaler()
        n_samples_train_y, n_timesteps_y, n_features_y = y_train.shape
        y_train_reshaped = y_train.reshape(-1, n_features_y)
        y_train_scaled_reshaped = target_scaler.fit_transform(y_train_reshaped)
        y_train_scaled = y_train_scaled_reshaped.reshape(n_samples_train_y, n_timesteps_y, n_features_y)

        n_samples_test_y = y_test.shape[0]
        y_test_reshaped = y_test.reshape(-1, n_features_y)
        y_test_scaled_reshaped = target_scaler.transform(y_test_reshaped)
        y_test_scaled = y_test_scaled_reshaped.reshape(n_samples_test_y, n_timesteps_y, n_features_y)
        
        train_dataset = QosDataset(X_train_scaled, y_train_scaled)
        test_dataset = QosDataset(X_test_scaled, y_test_scaled)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        client_loaders.append({
            'train': train_loader,
            'test': test_loader,
            'target_scaler': target_scaler,
            'feature_scaler' : scaler
        })

    return client_loaders

In [93]:
client_loaders = create_federated_data(
    all_windows=all_windows,
    all_targets=all_targets,
    num_clients=NUM_CLIENTS,
    batch_size = BATCH_SIZE
)

In [94]:
# global_test_loader = client_loaders[int(cid)]["train"]
global_test_loader = client_loaders[0]["test"]
global_target_scaler = client_loaders[0]["target_scaler"]

In [95]:
from typing import Dict, List, Optional, Tuple
from collections import OrderedDict

def get_parameters(model) -> List[np.ndarray]:
    return [val.cpu().numpy() for _, val in model.state_dict().items()]

def set_parameters(model, parameters : List[np.ndarray]):
    param_dict = zip(model.state_dict().keys(), parameters)
    state_dict = OrderedDict({k : torch.from_numpy(v) for k, v in param_dict})
    model.load_state_dict(state_dict, strict=True)

In [96]:
import torch
import torch.nn as nn

class LSTM(nn.Module):
    def __init__(self, input_size=12, num_target_features=2, future_window_len=10, hidden_size=50, num_layers=3, dropout=0.3, bidirectional=False):
        super(LSTM, self).__init__()
        self.num_target_features = num_target_features
        self.future_window_len = future_window_len
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.lstm = nn.LSTM(
            input_size, 
            hidden_size, 
            num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        direction_factor = 2 if bidirectional else 1
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * direction_factor, 128),
            nn.ReLU(),
            nn.Linear(128, future_window_len * num_target_features)
        )

    def forward(self, x):
        h0 = torch.zeros(self.num_layers * (2 if self.bidirectional else 1), x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * (2 if self.bidirectional else 1), x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :] 
        out = self.fc(out)
        out = out.view(x.size(0), self.future_window_len, self.num_target_features)
        return out

In [ ]:
def train(model, trainloader, criterion, optimizer, epochs, scheduler=None):
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0
        for X, y in trainloader:
            model.to(DEVICE)
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        if scheduler is not None:
            scheduler.step()
    
    avg_loss = epoch_loss / len(trainloader)
    
    return avg_loss

In [98]:
from sklearn.metrics import r2_score
import torch
import numpy as np

def validate(model, valloader, criterion, target_scaler):
    model.eval()
    epoch_loss = 0
    all_pred_unscaled = []
    all_targets_unscaled = []

    with torch.no_grad():
        for X,y in valloader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X)

            loss = criterion(pred, y)
            epoch_loss += loss.item()

            pred_cpu = pred.cpu().numpy()
            y_cpu = y.cpu().numpy()

            num_target_features = pred_cpu.shape[-1]

            pred_reshaped = pred_cpu.reshape(-1, num_target_features)
            y_reshaped = y_cpu.reshape(-1, num_target_features)

            pred_unscaled = target_scaler.inverse_transform(pred_reshaped)
            y_unscaled = target_scaler.inverse_transform(y_reshaped)

            all_pred_unscaled.append(pred_unscaled)
            all_targets_unscaled.append(y_unscaled)

    avg_loss = epoch_loss / len(valloader)

    final_preds = np.concatenate(all_pred_unscaled)
    final_targets = np.concatenate(all_targets_unscaled)

    num_features = final_targets.shape[1]
    r2_scores = {}

    for i in range(num_features):
        r2 = r2_score(final_targets[:, i], final_preds[:, i])
        r2_scores[f'r2_feature_{i}'] = r2

    return avg_loss, r2_scores

In [99]:
class QosClient(fl.client.NumPyClient):
    """
    A Flower client that handles local training and evaluation.
    """
    def __init__(self, cid: str, model: nn.Module, trainloader, testloader, 
                 target_scaler: MinMaxScaler, target_feature_names: list[str]):
        
        self.cid = cid
        self.model = model.to(DEVICE)
        self.trainloader = trainloader
        self.testloader = testloader
        self.target_scaler = target_scaler
        self.target_feature_names = target_feature_names
        
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=LEARNING_RATE)
        
        self.last_train_loss: float = -1.0

    def get_parameters(self, config: dict) -> list[np.ndarray]:
        """Returns the current local model parameters."""
        return get_parameters(self.model)

    def fit(self, parameters: list[np.ndarray], config: dict) -> tuple:
        """Trains the model and stores the final training loss."""
        set_parameters(self.model, parameters)
        avg_train_loss = train(
            self.model, self.trainloader, self.criterion, self.optimizer, config["local_epochs"]
        )
        
        self.last_train_loss = avg_train_loss
        
        return get_parameters(self.model), len(self.trainloader.dataset), {"train_loss": avg_train_loss}

    def evaluate(self, parameters: list[np.ndarray], config: dict) -> tuple:
        """Evaluates the model on local test data."""
        server_round = config["server_round"]
        set_parameters(self.model, parameters)
        
        val_loss, r2_scores = validate(
            self.model, self.testloader, self.criterion, self.target_scaler
        )
        # print(f"Client {self.cid} Round {server_round} | Train Loss: {self.last_train_loss:.4f} | Val Loss: {val_loss:.4f} | R²: {r2_scores}")
        return float(val_loss), len(self.testloader.dataset), r2_scores

In [100]:
def client_fn(cid) -> QosClient:
    model = LSTM(input_size=INPUT_SIZE, num_target_features=2, future_window_len=FUTURE_WIN_LEN, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS).to(DEVICE)
    trainloader = client_loaders[int(cid)]["train"]
    testloader = client_loaders[int(cid)]["test"]
    scaler = client_loaders[int(cid)]["target_scaler"]
    client = QosClient(cid, model, trainloader, testloader, scaler, target_feature_names)
    # return client.to_client()
    return client

In [111]:
import flwr as fl
import torch
import torch.nn as nn
from typing import Dict, List, Optional, Tuple
from flwr.common import Context

def run_single_experiment(
    hyperparams: dict, 
    client_loaders: list, 
    global_test_loader, 
    global_target_scaler, 
    target_feature_names: list
) -> dict:
    """
    Runs a single, complete federated learning experiment with a given set of 
    hyperparameters and returns the key performance metrics.

    Args:
        hyperparams (dict): A dictionary containing all hyperparameters for the run.
        client_loaders (list): The list of pre-processed client dataloaders.
        global_test_loader: The pre-processed global test dataloader.
        global_target_scaler: The scaler fitted on the training data for the global test set.
        target_feature_names (list): Names of the target features (e.g., ['throughput', 'delay']).

    Returns:
        dict: A dictionary containing the best R² scores and their corresponding rounds.
    """
    INPUT_SIZE = 4
    OUTPUT_SIZE = 2 

    dataset, all_windows, all_targets = create_autoregressive_dataset(
        df, cols, indices, future_window_len=hyperparams['FUTURE_WIN_LEN'], window_len=hyperparams['WINDOW_LEN'], stride=hyperparams['FUTURE_WIN_LEN']
    )

    client_loaders = create_federated_data(
        all_windows=all_windows,
        all_targets=all_targets,
        num_clients=NUM_CLIENTS,
        batch_size = hyperparams['BATCH_SIZE']
    )
    
    def get_evaluate_fn(test_loader, scaler):
        def evaluate(server_round: int, parameters: fl.common.NDArrays, config: dict):
            model = LSTM(
                input_size=hyperparams["INPUT_SIZE"],
                num_target_features=len(target_feature_names),
                future_window_len=hyperparams["FUTURE_WIN_LEN"],
                hidden_size=hyperparams["HIDDEN_SIZE"],
                num_layers=hyperparams["NUM_LAYERS"]
            ).to(DEVICE)
            set_parameters(model, parameters)
            loss, r2_scores = validate(model, test_loader, nn.MSELoss(), scaler)
            return loss, r2_scores
        return evaluate

    def client_fn(context: Context) -> fl.client.Client:
        cid = str(context.partition_id)
        model = LSTM(
            input_size=hyperparams["INPUT_SIZE"],
            num_target_features=len(target_feature_names),
            future_window_len=hyperparams["FUTURE_WIN_LEN"],
            hidden_size=hyperparams["HIDDEN_SIZE"],
            num_layers=hyperparams["NUM_LAYERS"]
        )
        client_data = client_loaders[int(cid)]
        return QosClient(
            cid, model, client_data['train'], client_data['test'],
            client_data['target_scaler'], target_feature_names
        ).to_client()

    initial_model = LSTM(
        input_size=INPUT_SIZE,
        num_target_features=len(target_feature_names),
        future_window_len=hyperparams["FUTURE_WIN_LEN"],
        hidden_size=hyperparams["HIDDEN_SIZE"],
        num_layers=hyperparams["NUM_LAYERS"]
    )

    initial_parameters = fl.common.ndarrays_to_parameters(get_parameters(initial_model))

    evaluate_fn = get_evaluate_fn(global_test_loader, global_target_scaler)
    
    def fit_config(server_round: int):
        return {"server_round": server_round, "local_epochs": hyperparams["NUM_EPOCHS"]}

    strategy = fl.server.strategy.FedAvg(
        fraction_fit=0.7,
        min_fit_clients=int(hyperparams["NUM_CLIENTS"] * 0.7),
        min_evaluate_clients=hyperparams["NUM_CLIENTS"],
        min_available_clients=hyperparams["NUM_CLIENTS"],
        initial_parameters=initial_parameters,
        evaluate_fn=evaluate_fn,
        on_fit_config_fn=fit_config,
    )

    history = fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=hyperparams["NUM_CLIENTS"],
        config=fl.server.ServerConfig(num_rounds=hyperparams["NUM_ROUNDS"]),
        strategy=strategy,
        client_resources={"num_cpus": 1, "num_gpus": 1 if DEVICE.type == "cuda" else 0}
    )
    
    final_results = {}
    for name in target_feature_names:
        metric_key = f'r2_{name}'
        if metric_key in history.metrics_centralized:
            best_entry = max(history.metrics_centralized[metric_key], key=lambda item: item[1])
            final_results[f'max_{metric_key}_round'] = best_entry[0]
            final_results[f'max_{metric_key}'] = best_entry[1]
    
    return final_results

In [112]:
hyperparams = {
    "BATCH_SIZE": 64,
    "LEARNING_RATE": 0.001,
    "NUM_EPOCHS": 250,
    "FUTURE_WIN_LEN": 10,
    "WINDOW_LEN": 50,
    "STRIDE": 1,
    "HIDDEN_SIZE": 256,
    "NUM_LAYERS": 2,
    "BATCH_SIZE": 64,
    "NUM_CLIENTS": 7,
    "NUM_ROUNDS": 200,
    "INPUT_SIZE": 4,
    "OUTPUT_SIZE": 2
}
results = run_single_experiment(hyperparams, client_loaders, global_test_loader, global_target_scaler, target_feature_names)
print(results)

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=200, no round_timeout
2025-08-28 22:23:46,068	INFO worker.py:1771 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'accelerator_type:G': 1.0, 'node:__internal_head__': 1.0, 'CPU': 16.0, 'object_store_memory': 1546203955.0, 'memory': 3092407911.0, 'node:192.168.49.157': 1.0, 'GPU': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0}
INFO :      F

RuntimeError: Simulation crashed.